# PINN to Solve Optimization Problems

**Tutorial:** Solving constrained optimisation problems by reformulating their KKT conditions as an IVP and solving it with Physics-Informed Neural Networks (PINNs) in JAX.

---

In [ ]:
# -----------------------------------------------------------------------
# Install dependencies
# -----------------------------------------------------------------------
# Google Colab: run the cell below as-is.
# Local (recommended): use uv for fast, reproducible installs:
#   uv venv && source .venv/bin/activate
#   uv pip install -r requirements.txt
# -----------------------------------------------------------------------
!pip install jax jaxlib flax optax plotly

## 1. Mathematical Background

### 1.1 Optimization Problem

We consider the standard non-linear programming (NLP) problem:

$$\min_{x \in \mathbb{R}^n} \quad f(x)$$

$$\text{subject to} \quad h_i(x) = 0, \quad i = 1, \dots, p$$

$$\quad\quad\quad\quad\quad\;\; g_j(x) \le 0, \quad j = 1, \dots, m$$

where $f : \mathbb{R}^n \to \mathbb{R}$ is the objective, $h : \mathbb{R}^n \to \mathbb{R}^p$ are equality constraints, and $g : \mathbb{R}^n \to \mathbb{R}^m$ are inequality constraints.

The **Lagrangian** combines the objective with multiplier-weighted constraint residuals:

$$L(x, \lambda, \mu) = f(x) + \lambda^\top h(x) + \mu^\top g(x)$$

### 1.2 KKT Conditions

At a local minimum $(x^*, \lambda^*, \mu^*)$ the **Karush-Kuhn-Tucker (KKT) conditions** must hold:

1. **Stationarity**: $\nabla_x L = 0$ — the gradient of $L$ w.r.t. $x$ vanishes at the optimum.
2. **Primal feasibility**: $h(x^*) = 0$, $g(x^*) \le 0$ — the constraints are satisfied.
3. **Dual feasibility**: $\mu^* \ge 0$ — inequality multipliers are non-negative.
4. **Complementary slackness**: $\mu^*_j g_j(x^*) = 0$ — either the constraint is active ($g_j = 0$) or its multiplier is zero.

### 1.3 IVP via Primal-Dual Gradient Flow

Instead of solving the KKT system as a static root-finding problem, we **embed it in a continuous-time ODE** whose equilibrium is the KKT point:

$$\frac{dx}{dt} = -\nabla_x L(x, \lambda, \mu) \quad \text{(primal descent)}$$

$$\frac{d\lambda}{dt} = h(x) \quad \text{(dual ascent — enforces equality)}$$

$$\frac{d\mu}{dt} = [g(x)]_+ \quad \text{(projected dual ascent — enforces inequality + } \mu \ge 0\text{)}$$

At steady state ($\dot{z} = 0$), all four KKT conditions are recovered. Pairing the ODE with **initial conditions** $z(0) = z_0$ gives a well-posed **Initial Value Problem (IVP)**:

$$\dot{z}(t) = F(z(t)), \qquad z(0) = z_0, \qquad t \in [0, T]$$

### 1.4 PINN Approach

Rather than using a classical numerical integrator (Euler, Runge-Kutta …), we **parameterise** the solution trajectory $z(t) = (x(t), \lambda(t))$ with a neural network $\hat{z}_\theta(t)$ and **minimise a physics-informed loss**:

$$\mathcal{L}(\theta) = \underbrace{\frac{1}{N}\sum_k \|\dot{\hat{z}}_\theta(t_k) - F(\hat{z}_\theta(t_k))\|^2}_{\mathcal{L}_{\text{ODE}}} + \underbrace{\|\hat{z}_\theta(0) - z_0\|^2}_{\mathcal{L}_{\text{IC}}}$$

**Key insight:** The time derivative $\dot{\hat{z}}_\theta(t_k)$ is computed **exactly** via `jax.grad` / `jax.jacfwd` — no finite differences, no discretisation error.

---
## 2. Toy Problem

$$\min_{x,y} \quad x^2 + y^2 \qquad \text{s.t.} \quad x + y = 1$$

Analytical solution: $x^* = y^* = 0.5$, $\lambda^* = -1$.

Below we define all problem functions using JAX.  Because JAX operations are **differentiable by default**, we can call `jax.grad` on any of them to obtain exact gradients — the foundation of both the gradient-flow dynamics and the PINN loss.

In [1]:
import jax
import jax.numpy as jnp

# ---- Objective function ------------------------------------------------

def objective(xy):
    """f(x, y) = x² + y² — convex quadratic to minimise.

    Args:
        xy: Array of shape (2,) containing [x, y].

    Returns:
        Scalar value of the objective.
    """
    x, y = xy[0], xy[1]
    return x**2 + y**2


# ---- Equality constraint -----------------------------------------------

def equality_constraint(xy):
    """h(x, y) = x + y − 1 (must equal 0 at a feasible point).

    Args:
        xy: Array of shape (2,).

    Returns:
        Scalar constraint residual.
    """
    x, y = xy[0], xy[1]
    return x + y - 1.0


# ---- Lagrangian --------------------------------------------------------

def lagrangian(xy, lam):
    """L(x, y, λ) = f(x, y) + λ·h(x, y).

    No inequality constraints in this problem, so no μ term.

    Args:
        xy: Array of shape (2,).
        lam: Array of shape (1,) — the equality multiplier.

    Returns:
        Scalar Lagrangian value.
    """
    return objective(xy) + lam[0] * equality_constraint(xy)


# ---- Gradient-flow RHS -------------------------------------------------

def primal_rhs(xy, lam):
    """dx/dt = −∇_x L  (primal gradient descent on the Lagrangian).

    ``jax.grad(lagrangian, argnums=0)`` differentiates lagrangian w.r.t.
    its *first* argument (xy), giving the exact gradient ∇_x L without
    any finite-difference approximation.

    Args:
        xy: Array of shape (2,).
        lam: Array of shape (1,).

    Returns:
        Array of shape (2,) representing dx/dt.
    """
    # argnums=0 → differentiate w.r.t. xy (the first positional argument)
    return -jax.grad(lagrangian, argnums=0)(xy, lam)


def dual_rhs(xy):
    """dλ/dt = h(x)  (dual gradient ascent enforcing primal feasibility).

    At steady state dλ/dt = 0 implies h(x) = 0, i.e. the constraint is
    satisfied.

    Args:
        xy: Array of shape (2,).

    Returns:
        Array of shape (1,) representing dλ/dt.
    """
    return jnp.array([equality_constraint(xy)])


ANALYTICAL_SOLUTION = {"x": 0.5, "y": 0.5, "lambda": -1.0}
print("Problem functions defined.  Analytical solution:", ANALYTICAL_SOLUTION)

Problem functions defined.  Analytical solution: {'x': 0.5, 'y': 0.5, 'lambda': -1.0}


---
## 3. Neural Network Architecture (Flax)

We use **Flax** (`flax.linen`) to define the network.  Flax is a functional neural-network library built on JAX:

* **Parameters live outside the model** — `model.apply({"params": params}, t)` does a *pure* forward pass with no hidden mutable state.  This makes the network safe to differentiate with `jax.grad`, batch with `jax.vmap`, and compile with `jax.jit`.
* **`@nn.compact`** lets us declare layers (e.g. `nn.Dense`) inline in `__call__` rather than in a separate `setup` method — cleaner for small networks.
* **`tanh` activations** are preferred over ReLU for PINNs because the network is differentiated w.r.t. its *input* $t$; tanh is smooth (infinitely differentiable) and avoids dead-neuron issues at the boundary.

In [2]:
import flax.linen as nn
from typing import Sequence

class PINN(nn.Module):
    """Fully-connected PINN: scalar time t → state vector [x(t), y(t), λ(t)].

    Attributes:
        hidden_sizes: Tuple of hidden-layer widths, e.g. (32, 32, 32).
    """
    hidden_sizes: Sequence[int] = (32, 32, 32)

    @nn.compact
    def __call__(self, t):
        """Forward pass.

        Args:
            t: Scalar time value.

        Returns:
            Array of shape (3,) — [x(t), y(t), λ(t)].
        """
        # Promote scalar t to shape (1,) so nn.Dense can process it
        z = jnp.atleast_1d(t)

        # Hidden layers: affine transform + tanh activation
        # tanh is smooth and bounded — ideal for differentiating through
        for size in self.hidden_sizes:
            z = nn.Dense(size)(z)   # z = W·z + b
            z = nn.tanh(z)

        z = nn.Dense(3)(z)          # linear readout → [x, y, λ]
        return z


# Instantiate and print a summary of the architecture
model = PINN(hidden_sizes=(32, 32, 32))
print(model.tabulate(jax.random.PRNGKey(0), jnp.array(0.0)))


                              PINN Summary                               
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs      ┃ outputs     ┃ params                 ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│         │ PINN   │ float32[]   │ float32[3]  │                        │
├─────────┼────────┼─────────────┼─────────────┼────────────────────────┤
│ Dense_0 │ Dense  │ float32[1]  │ float32[32] │ bias: float32[32]      │
│         │        │             │             │ kernel: float32[1,32]  │
│         │        │             │             │                        │
│         │        │             │             │ 64 (256 B)             │
├─────────┼────────┼─────────────┼─────────────┼────────────────────────┤
│ Dense_1 │ Dense  │ float32[32] │ float32[32] │ bias: float32[32]      │
│         │        │             │             │ kernel: float32[32,32] │
│         │        │             │   

---
## 4. PINN Loss Function

The loss penalises two types of violations:

| Term | What it enforces | Key JAX tool |
|---|---|---|
| $\mathcal{L}_{\text{ODE}}$ | ODE residual $\dot{\hat{z}} = F(\hat{z})$ at collocation points | `jax.jacfwd` + `jax.vmap` |
| $\mathcal{L}_{\text{IC}}$ | Initial condition $\hat{z}(0) = z_0$ | plain evaluation |

**`jax.jacfwd(net)`** computes the *exact* Jacobian $d\hat{z}/dt$ using forward-mode automatic differentiation.  Forward-mode is efficient here because the input is 1-D while the output is 3-D (one forward pass yields all three derivatives simultaneously).

**`jax.vmap(ode_residual)`** vectorises the residual computation over all $N$ collocation points, producing an $(N, 3)$ array in a single fused XLA kernel — no Python loop needed.

In [3]:
from functools import partial

def pinn_loss(params, apply_fn, t_colloc, z0, w_ode=1.0, w_ic=10.0):
    """Computes the total physics-informed loss.

    Args:
        params: Flax parameter pytree.
        apply_fn: model.apply — pure functional forward pass.
        t_colloc: Collocation time points, shape (N,).
        z0: Initial state [x0, y0, λ0], shape (3,).
        w_ode: Weight for ODE residual term.
        w_ic: Weight for initial-condition term (higher = stricter IC).

    Returns:
        Scalar loss value.
    """

    # Closure: net(t) is a pure function of t given fixed params
    def net(t):
        return apply_fn({"params": params}, t)

    # jax.jacfwd: forward-mode AD — efficient for scalar input, vector output
    # Returns dnet(t)/dt, a (3,) vector of exact time derivatives
    dnet_dt_vec = jax.jacfwd(net)

    def ode_residual(t):
        """ODE residual at a single time point: dẑ/dt − F(ẑ)."""
        z  = net(t)                     # network prediction [x, y, λ]
        dz = dnet_dt_vec(t).squeeze()   # exact dẑ/dt via AD, shape (3,)

        # Unpack into primal and dual parts
        xy  = z[:2]                     # [x, y]
        lam = z[2:3]                    # [λ]

        # Evaluate gradient-flow RHS from the problem definition
        rhs = jnp.concatenate([
            primal_rhs(xy, lam),        # −∇_x L, shape (2,)
            dual_rhs(xy),               #  h(x),  shape (1,)
        ])                              # combined RHS, shape (3,)

        return dz - rhs                 # zero when ODE is satisfied

    # jax.vmap: map ode_residual over the N collocation points at once
    # Equivalent to [ode_residual(t) for t in t_colloc] but compiled as
    # a single batched XLA operation — no Python for-loop overhead
    residuals = jax.vmap(ode_residual)(t_colloc)   # shape (N, 3)
    L_ode = jnp.mean(residuals ** 2)

    # Initial-condition loss: penalise deviation from z0 at t = 0
    z_pred_0 = net(jnp.array(0.0))
    L_ic = jnp.mean((z_pred_0 - z0) ** 2)

    return w_ode * L_ode + w_ic * L_ic


print("Loss function defined.")

Loss function defined.


---
## 5. Training Loop

We use **Optax** (`optax.adam`) as the gradient-based optimiser.  The training loop follows the standard JAX pattern:

1. `jax.value_and_grad(loss)(params)` — one forward + backward pass returning both the loss and its gradient $\partial \mathcal{L}/\partial \theta$.
2. `optimizer.update(grads, opt_state)` — Optax computes Adam-scaled parameter updates.
3. `optax.apply_updates(params, updates)` — applies the updates to the parameter pytree.

`jax.jit` wraps the entire `value_and_grad` call, compiling it once to XLA and eliminating Python overhead on every subsequent step.

In [4]:
import optax

# ---- Hyperparameters --------------------------------------------------
T         = 5.0     # pseudo-time horizon for the gradient-flow IVP
N_COLLOC  = 100     # collocation points (uniformly spaced in [0, T])
N_EPOCHS  = 5000    # total gradient-descent steps
LR        = 1e-3    # Adam learning rate
SEED      = 42      # PRNG seed for reproducibility

# Initial conditions z(0) = [x0, y0, λ0] — arbitrary starting point
Z0 = jnp.array([0.0, 2.0, 0.0])

# ---- Collocation points -----------------------------------------------
t_colloc = jnp.linspace(0.0, T, N_COLLOC)

# ---- Initialise network parameters ------------------------------------
# JAX has no global random state; all randomness flows through explicit keys.
# model.init does a single forward pass to infer shapes, then initialises
# all Dense weights/biases randomly using the provided key.
key = jax.random.PRNGKey(SEED)
params = model.init(key, jnp.array(0.0))["params"]

# ---- Optax Adam optimiser ---------------------------------------------
optimizer = optax.adam(LR)
opt_state = optimizer.init(params)   # initialise momentum buffers etc.

# ---- JIT-compiled loss + gradient -------------------------------------
# jax.value_and_grad: computes loss AND its gradient in one pass.
# partial pins apply_fn, t_colloc, z0 — only params is differentiated.
# jax.jit: compiles the whole forward+backward pass to XLA for speed.
loss_and_grad = jax.jit(
    jax.value_and_grad(
        partial(pinn_loss, apply_fn=model.apply, t_colloc=t_colloc, z0=Z0)
    )
)

# ---- Training loop ----------------------------------------------------
loss_history = []

for epoch in range(1, N_EPOCHS + 1):
    # Forward + backward pass
    loss_val, grads = loss_and_grad(params)

    # Optax: compute Adam updates from gradients and optimiser state
    updates, opt_state = optimizer.update(grads, opt_state)

    # Apply updates: θ ← θ + Δθ  (Δθ = Adam-scaled gradient step)
    params = optax.apply_updates(params, updates)

    loss_history.append(float(loss_val))
    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d} | Loss {loss_val:.6e}")

print("\nTraining complete.")

Epoch   500 | Loss 4.016119e-03
Epoch  1000 | Loss 1.327781e-03
Epoch  1500 | Loss 7.146536e-04
Epoch  2000 | Loss 4.241351e-04
Epoch  2500 | Loss 2.727225e-04
Epoch  3000 | Loss 1.908220e-04
Epoch  3500 | Loss 1.377932e-04
Epoch  4000 | Loss 1.058952e-04
Epoch  4500 | Loss 7.530875e-04
Epoch  5000 | Loss 6.662914e-05

Training complete.


---
## 6. Results and Interactive Visualisation

After training, we evaluate the network on a fine time grid $t \in [0, T]$ and plot:

1. **Loss curve** — should decrease monotonically, confirming that the ODE residual and initial-condition errors are being driven to zero.
2. **Trajectories** — the primal variables $(x(t), y(t))$ and dual variable $\lambda(t)$ should converge to the analytical KKT solution $(0.5, 0.5, -1)$ as $t \to T$.

The plots use **Plotly** for interactive zoom, hover, and export.

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Evaluate on a fine time grid -------------------------------------
# jax.vmap vectorises the network call over all 300 time points at once —
# no Python loop, compiled into a single batched XLA operation.
t_eval = jnp.linspace(0.0, T, 300)
trajectories = jax.vmap(
    lambda t: model.apply({"params": params}, t)
)(t_eval)                           # shape (300, 3)

x_traj   = trajectories[:, 0]      # primal x(t)
y_traj   = trajectories[:, 1]      # primal y(t)
lam_traj = trajectories[:, 2]      # dual   λ(t)

print("Final PINN values (at t = T):")
print(f"  x      = {float(x_traj[-1]):.6f}   (analytical: 0.5)")
print(f"  y      = {float(y_traj[-1]):.6f}   (analytical: 0.5)")
print(f"  lambda = {float(lam_traj[-1]):.6f}   (analytical: -1.0)")

# Helper: convert JAX arrays to plain Python lists for Plotly
t_np = [float(v) for v in t_eval]

# ---- Plot 1: Training loss curve (log scale) --------------------------
epochs = list(range(1, len(loss_history) + 1))
fig_loss = go.Figure(
    go.Scatter(x=epochs, y=loss_history, mode="lines",
               name="Loss", line=dict(color="crimson"))
)
fig_loss.update_layout(
    title="PINN Training Loss",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    yaxis_type="log",    # log scale makes multi-order-of-magnitude drop visible
)
fig_loss.show()

# ---- Plot 2: Primal & dual trajectories --------------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Primal variable x", "Primal variable y", "Dual variable λ"),
)
fig.update_layout(title_text="PINN Trajectories Converging to the KKT Point")

# Primal x — should converge to x* = 0.5
fig.add_trace(
    go.Scatter(x=t_np, y=[float(v) for v in x_traj],
               mode="lines", name="x̂(t)", line=dict(color="steelblue")),
    row=1, col=1,
)
fig.add_hline(y=0.5, line_dash="dash", line_color="steelblue",
              annotation_text="x*=0.5", row=1, col=1)

# Primal y — should converge to y* = 0.5
fig.add_trace(
    go.Scatter(x=t_np, y=[float(v) for v in y_traj],
               mode="lines", name="ŷ(t)", line=dict(color="darkorange")),
    row=1, col=2,
)
fig.add_hline(y=0.5, line_dash="dash", line_color="darkorange",
              annotation_text="y*=0.5", row=1, col=2)

# Dual λ — should converge to λ* = -1
fig.add_trace(
    go.Scatter(x=t_np, y=[float(v) for v in lam_traj],
               mode="lines", name="λ̂(t)", line=dict(color="seagreen")),
    row=1, col=3,
)
fig.add_hline(y=-1.0, line_dash="dash", line_color="seagreen",
              annotation_text="λ*=−1", row=1, col=3)

fig.update_xaxes(title_text="t")
fig.show()

Final PINN values (at t = T):
  x      = 0.515911   (analytical: 0.5)
  y      = 0.514234   (analytical: 0.5)
  lambda = -1.010378   (analytical: -1.0)
